# Smart Data Preprocessing Pipeline
## Complete Backend Logic for Data Quality Dashboard

## SECTION 1: Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sqlite3
import os
import io
import re
import json
import joblib
import warnings
from datetime import datetime
from scipy import stats
from scipy.stats import boxcox
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from difflib import SequenceMatcher
import openpyxl

warnings.filterwarnings('ignore')
print('All imports successful.')

All imports successful.


## SECTION 2: Database Setup

In [2]:
DB_NAME = 'dashboard.db'

def init_database():
    """Initialize SQLite database and create required tables."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS file_metadata (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_name TEXT,
            upload_date TEXT,
            file_size_kb REAL,
            row_count INTEGER,
            column_count INTEGER
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS processing_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_name TEXT,
            operation TEXT,
            details TEXT,
            timestamp TEXT
        )
    ''')

    conn.commit()
    conn.close()
    print(f'Database "{DB_NAME}" initialized successfully.')

init_database()

Database "dashboard.db" initialized successfully.


## SECTION 3: File Loader Functions

In [3]:
def load_file(file_path_or_buffer, file_name=''):
    """Load CSV, XLSX, or XLS file into a DataFrame."""
    try:
        name = file_name.lower() if file_name else str(file_path_or_buffer).lower()
        if name.endswith('.csv'):
            df = pd.read_csv(file_path_or_buffer, low_memory=False)
        elif name.endswith('.xlsx'):
            df = pd.read_excel(file_path_or_buffer, engine='openpyxl')
        elif name.endswith('.xls'):
            df = pd.read_excel(file_path_or_buffer, engine='xlrd')
        else:
            raise ValueError(f'Unsupported file format: {name}')
        print(f'Loaded {len(df)} rows x {len(df.columns)} columns.')
        return df
    except Exception as e:
        print(f'Error loading file: {e}')
        return None


def save_uploaded_file(uploaded_file, save_dir='uploaded_files'):
    """Save uploaded Streamlit file to disk."""
    try:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, uploaded_file.name)
        with open(save_path, 'wb') as f:
            f.write(uploaded_file.getbuffer())
        print(f'File saved to: {save_path}')
        return save_path
    except Exception as e:
        print(f'Error saving file: {e}')
        return None


def store_metadata(file_name, file_size_kb, row_count, col_count):
    """Store file metadata in SQLite database."""
    try:
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO file_metadata (file_name, upload_date, file_size_kb, row_count, column_count)
            VALUES (?, ?, ?, ?, ?)
        ''', (file_name, datetime.now().isoformat(), file_size_kb, row_count, col_count))
        conn.commit()
        conn.close()
        print('Metadata stored successfully.')
    except Exception as e:
        print(f'Error storing metadata: {e}')

## SECTION 4: Data Inspection Functions

In [4]:
def get_dataset_summary(df):
    """Return a high-level summary of the dataset."""
    try:
        summary = {
            'rows': len(df),
            'columns': len(df.columns),
            'total_cells': df.size,
            'missing_cells': int(df.isnull().sum().sum()),
            'missing_pct': round(df.isnull().sum().sum() / df.size * 100, 2),
            'duplicate_rows': int(df.duplicated().sum()),
            'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 3),
            'dtypes': df.dtypes.astype(str).to_dict(),
            'columns_list': list(df.columns)
        }
        return summary
    except Exception as e:
        return {'error': str(e)}


def identify_column_types(df):
    """Auto-detect column types: numerical, categorical, datetime, boolean, id."""
    col_types = {'numerical': [], 'categorical': [], 'datetime': [], 'boolean': [], 'id': []}
    for col in df.columns:
        col_lower = col.lower()
        series = df[col].dropna()
        if series.empty:
            col_types['categorical'].append(col)
            continue
        # ID detection
        if any(kw in col_lower for kw in ['id', '_id', 'index', 'key', 'uuid', 'guid']):
            if series.nunique() == len(series):
                col_types['id'].append(col)
                continue
        # Boolean
        if series.dtype == bool or set(series.unique()).issubset({True, False, 0, 1, 'true', 'false', 'True', 'False', 'yes', 'no', 'Yes', 'No'}):
            col_types['boolean'].append(col)
            continue
        # Datetime
        if pd.api.types.is_datetime64_any_dtype(series):
            col_types['datetime'].append(col)
            continue
        if series.dtype == object:
            try:
                pd.to_datetime(series.head(50), infer_datetime_format=True)
                col_types['datetime'].append(col)
                continue
            except:
                pass
        # Numerical
        if pd.api.types.is_numeric_dtype(series):
            col_types['numerical'].append(col)
        else:
            col_types['categorical'].append(col)
    return col_types


def memory_usage(df):
    """Return per-column memory usage in KB."""
    mem = df.memory_usage(deep=True)
    mem_df = pd.DataFrame({'Column': mem.index, 'Memory_KB': (mem.values / 1024).round(3)})
    return mem_df


def missing_value_summary(df):
    """Return missing value counts and percentages per column."""
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    result = pd.DataFrame({'Column': missing.index, 'Missing_Count': missing.values, 'Missing_Pct': pct.values})
    return result[result['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)


def duplicate_summary(df):
    """Return duplicate rows."""
    dupes = df[df.duplicated(keep='first')]
    return {'duplicate_count': len(dupes), 'duplicate_rows': dupes}

## SECTION 5: Dynamic User Data Entry

In [5]:
def generate_input_schema(df):
    """Generate dynamic input schema based on detected column types."""
    col_types = identify_column_types(df)
    schema = {}
    for col in df.columns:
        entry = {'column': col, 'dtype': str(df[col].dtype)}
        if col in col_types['numerical']:
            entry['input_type'] = 'number'
            entry['min'] = float(df[col].min()) if not df[col].isna().all() else None
            entry['max'] = float(df[col].max()) if not df[col].isna().all() else None
        elif col in col_types['categorical']:
            entry['input_type'] = 'select'
            entry['options'] = list(df[col].dropna().unique())
        elif col in col_types['datetime']:
            entry['input_type'] = 'date'
        elif col in col_types['boolean']:
            entry['input_type'] = 'boolean'
            entry['options'] = [True, False]
        else:
            entry['input_type'] = 'text'
        schema[col] = entry
    return schema


def validate_user_input(df, input_data):
    """Validate user-provided row against schema rules."""
    schema = generate_input_schema(df)
    errors = {}
    for col, spec in schema.items():
        val = input_data.get(col)
        if val is None or val == '':
            continue
        try:
            if spec['input_type'] == 'number':
                fval = float(val)
                if spec.get('min') is not None and fval < spec['min']:
                    errors[col] = f'Value {fval} below min {spec["min"]}'
                if spec.get('max') is not None and fval > spec['max']:
                    errors[col] = f'Value {fval} above max {spec["max"]}'
            elif spec['input_type'] == 'select':
                if val not in spec['options']:
                    errors[col] = f'Invalid value. Choose from: {spec["options"][:5]}'
            elif spec['input_type'] == 'date':
                pd.to_datetime(val)
            elif spec['input_type'] == 'boolean':
                if str(val).lower() not in ['true', 'false', '1', '0', 'yes', 'no']:
                    errors[col] = 'Must be True or False'
        except Exception as e:
            errors[col] = str(e)
    return errors


def add_new_row(df, input_data):
    """Append a new validated row to the DataFrame."""
    errors = validate_user_input(df, input_data)
    if errors:
        return df, errors
    new_row = {col: input_data.get(col, np.nan) for col in df.columns}
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    return df, {}

## SECTION 6: Duplicate Handling

In [6]:
def find_duplicates(df):
    """Identify duplicate rows."""
    dupes = df[df.duplicated(keep='first')]
    return {'count': len(dupes), 'rows': dupes, 'pct': round(len(dupes)/len(df)*100, 2)}


def remove_duplicates(df):
    """Remove duplicate rows and return cleaned DataFrame."""
    original_shape = df.shape
    df_clean = df.drop_duplicates(keep='first').reset_index(drop=True)
    removed = original_shape[0] - df_clean.shape[0]
    return df_clean, {'original_rows': original_shape[0], 'removed': removed, 'new_shape': df_clean.shape}

## SECTION 7: Missing Value Handling

In [7]:
def missing_value_report(df):
    """Detailed missing value report."""
    col_types = identify_column_types(df)
    report = []
    for col in df.columns:
        missing = df[col].isnull().sum()
        if missing > 0:
            if col in col_types['numerical']:
                dtype = 'numerical'
                strategies = ['mean', 'median', 'mode', 'drop']
            elif col in col_types['datetime']:
                dtype = 'datetime'
                strategies = ['ffill', 'bfill', 'drop']
            else:
                dtype = 'categorical'
                strategies = ['mode', 'ffill', 'bfill', 'custom', 'drop']
            report.append({'column': col, 'missing': missing, 'pct': round(missing/len(df)*100,2), 'type': dtype, 'strategies': strategies})
    return report


def fill_missing_values(df, column, strategy, custom_value=None):
    """Fill missing values for a specific column using chosen strategy."""
    before = df[column].isnull().sum()
    df = df.copy()
    if strategy == 'mean':
        df[column].fillna(df[column].mean(), inplace=True)
    elif strategy == 'median':
        df[column].fillna(df[column].median(), inplace=True)
    elif strategy == 'mode':
        mode_val = df[column].mode()
        if not mode_val.empty:
            df[column].fillna(mode_val[0], inplace=True)
    elif strategy == 'ffill':
        df[column].fillna(method='ffill', inplace=True)
    elif strategy == 'bfill':
        df[column].fillna(method='bfill', inplace=True)
    elif strategy == 'custom' and custom_value is not None:
        df[column].fillna(custom_value, inplace=True)
    elif strategy == 'drop':
        df = df.dropna(subset=[column]).reset_index(drop=True)
    after = df[column].isnull().sum()
    return df, {'column': column, 'strategy': strategy, 'before': before, 'after': after, 'filled': before - after}


def drop_missing_rows(df, threshold=0.5):
    """Drop rows where missing values exceed threshold fraction."""
    before = len(df)
    df_clean = df.dropna(thresh=int(len(df.columns) * (1 - threshold))).reset_index(drop=True)
    return df_clean, {'before': before, 'after': len(df_clean), 'dropped': before - len(df_clean)}

## SECTION 8: Data Validation

In [8]:
def detect_invalid_values(df):
    """Detect domain-specific invalid values using column name heuristics."""
    issues = {}
    col_lower_map = {col: col.lower() for col in df.columns}
    for col, col_lower in col_lower_map.items():
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        if any(k in col_lower for k in ['age']):
            bad = df[(df[col] < 0) | (df[col] > 150)]
            if len(bad): issues[col] = {'issue': 'Age out of range (0-150)', 'rows': bad.index.tolist(), 'count': len(bad)}
        elif any(k in col_lower for k in ['pct', 'percent', 'rate', 'ratio']):
            bad = df[(df[col] < 0) | (df[col] > 100)]
            if len(bad): issues[col] = {'issue': 'Percentage out of range (0-100)', 'rows': bad.index.tolist(), 'count': len(bad)}
        elif any(k in col_lower for k in ['salary', 'income', 'revenue', 'price', 'cost', 'amount']):
            bad = df[df[col] < 0]
            if len(bad): issues[col] = {'issue': 'Negative monetary value', 'rows': bad.index.tolist(), 'count': len(bad)}
        elif any(k in col_lower for k in ['experience', 'exp', 'tenure']):
            bad = df[df[col] < 0]
            if len(bad): issues[col] = {'issue': 'Negative experience value', 'rows': bad.index.tolist(), 'count': len(bad)}
    return issues


def detect_negative_values(df):
    """Detect negative values in all numerical columns."""
    result = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        neg = df[df[col] < 0]
        if len(neg) > 0:
            result[col] = {'count': len(neg), 'rows': neg.index.tolist()}
    return result


def detect_invalid_email(df):
    """Detect invalid emails in columns that look like email fields."""
    email_pattern = re.compile(r'^[\w\.-]+@[\w\.-]+\.\w{2,}$')
    result = {}
    for col in df.select_dtypes(include='object').columns:
        if any(k in col.lower() for k in ['email', 'mail', 'e-mail']):
            bad = df[~df[col].dropna().apply(lambda x: bool(email_pattern.match(str(x))))]
            if len(bad) > 0:
                result[col] = {'count': len(bad), 'rows': bad.index.tolist()}
    return result


def detect_invalid_phone(df):
    """Detect invalid phone numbers in phone-related columns."""
    phone_pattern = re.compile(r'^[\+]?[\d\s\-\(\)]{7,15}$')
    result = {}
    for col in df.select_dtypes(include='object').columns:
        if any(k in col.lower() for k in ['phone', 'mobile', 'tel', 'contact']):
            bad = df[~df[col].dropna().apply(lambda x: bool(phone_pattern.match(str(x))))]
            if len(bad) > 0:
                result[col] = {'count': len(bad), 'rows': bad.index.tolist()}
    return result


def detect_future_dates(df):
    """Detect future dates in date-related columns."""
    now = pd.Timestamp.now()
    result = {}
    for col in df.columns:
        if any(k in col.lower() for k in ['birth', 'dob', 'born', 'date', 'created', 'joined']):
            try:
                parsed = pd.to_datetime(df[col], errors='coerce')
                future = df[parsed > now]
                if len(future) > 0:
                    result[col] = {'count': len(future), 'rows': future.index.tolist()}
            except:
                pass
    return result

## SECTION 9: Similar Column Detection

In [9]:
def detect_duplicate_information_columns(df):
    """Detect columns that likely contain duplicate/redundant information."""
    suggestions = []
    cols = df.columns.tolist()
    num_df = df.select_dtypes(include=[np.number])

    # Name similarity
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            ratio = SequenceMatcher(None, cols[i].lower(), cols[j].lower()).ratio()
            if ratio > 0.7:
                suggestions.append({'col1': cols[i], 'col2': cols[j], 'reason': f'Name similarity: {ratio:.2f}', 'score': ratio})

    # High correlation for numerical columns
    if len(num_df.columns) > 1:
        corr = num_df.corr().abs()
        for i in range(len(corr.columns)):
            for j in range(i+1, len(corr.columns)):
                val = corr.iloc[i, j]
                if val > 0.98:
                    suggestions.append({'col1': corr.columns[i], 'col2': corr.columns[j], 'reason': f'Near-perfect correlation: {val:.3f}', 'score': float(val)})

    # Unique value mapping (one-to-one)
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    for i in range(len(cat_cols)):
        for j in range(i+1, len(cat_cols)):
            try:
                mapping = df[[cat_cols[i], cat_cols[j]]].dropna().drop_duplicates()
                if len(mapping) == df[cat_cols[i]].nunique() and len(mapping) == df[cat_cols[j]].nunique():
                    suggestions.append({'col1': cat_cols[i], 'col2': cat_cols[j], 'reason': 'One-to-one categorical mapping', 'score': 0.95})
            except:
                pass

    return suggestions

## SECTION 10: Encoding

In [10]:
def detect_encoding_candidates(df):
    """Detect columns that are candidates for encoding."""
    col_types = identify_column_types(df)
    candidates = []
    for col in col_types['categorical'] + col_types['boolean']:
        n_unique = df[col].nunique()
        candidates.append({'column': col, 'unique_values': n_unique, 'sample': list(df[col].dropna().unique()[:5])})
    return candidates


def recommend_encoding(df, column):
    """Recommend encoding strategy based on cardinality."""
    n_unique = df[column].nunique()
    if n_unique == 2:
        return 'label'
    elif n_unique <= 10:
        return 'onehot'
    elif n_unique <= 50:
        return 'frequency'
    else:
        return 'frequency'


def apply_encoding(df, column, encoding_type, ordinal_order=None):
    """Apply selected encoding to a column."""
    df = df.copy()
    mapping_table = None
    before_cols = list(df.columns)

    if encoding_type == 'label':
        le = LabelEncoder()
        df[column + '_encoded'] = le.fit_transform(df[column].astype(str))
        mapping_table = pd.DataFrame({'Original': le.classes_, 'Encoded': range(len(le.classes_))})

    elif encoding_type == 'onehot':
        dummies = pd.get_dummies(df[column], prefix=column)
        df = pd.concat([df, dummies], axis=1)
        mapping_table = pd.DataFrame({'Original': list(dummies.columns), 'Encoded': list(dummies.columns)})

    elif encoding_type == 'ordinal' and ordinal_order:
        order_map = {v: i for i, v in enumerate(ordinal_order)}
        df[column + '_ordinal'] = df[column].map(order_map)
        mapping_table = pd.DataFrame({'Original': ordinal_order, 'Encoded': range(len(ordinal_order))})

    elif encoding_type == 'frequency':
        freq = df[column].value_counts(normalize=True)
        df[column + '_freq'] = df[column].map(freq)
        mapping_table = freq.reset_index()
        mapping_table.columns = ['Original', 'Frequency']

    after_cols = list(df.columns)
    new_cols = [c for c in after_cols if c not in before_cols]
    return df, mapping_table, new_cols

## SECTION 11: Outlier Detection

In [11]:
def detect_outliers_iqr(df):
    """Detect outliers using IQR method for all numerical columns."""
    result = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower) | (df[col] > upper)]
        result[col] = {
            'count': len(outliers),
            'pct': round(len(outliers) / len(df) * 100, 2),
            'lower_bound': lower,
            'upper_bound': upper,
            'outlier_rows': outliers.index.tolist()
        }
    return result


def detect_outliers_zscore(df, threshold=3):
    """Detect outliers using Z-score method."""
    result = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        z_scores = np.abs(stats.zscore(df[col].dropna()))
        outlier_mask = z_scores > threshold
        valid_index = df[col].dropna().index
        outlier_idx = valid_index[outlier_mask].tolist()
        result[col] = {
            'count': len(outlier_idx),
            'pct': round(len(outlier_idx) / len(df) * 100, 2),
            'outlier_rows': outlier_idx
        }
    return result


def remove_outliers(df, column, method='iqr'):
    """Remove outlier rows from a specific column."""
    before = len(df)
    if method == 'iqr':
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        df = df[(df[column] >= Q1 - 1.5*IQR) & (df[column] <= Q3 + 1.5*IQR)]
    elif method == 'zscore':
        z = np.abs(stats.zscore(df[column].dropna()))
        valid_idx = df[column].dropna().index[z <= 3]
        df = df.loc[valid_idx]
    df = df.reset_index(drop=True)
    return df, {'column': column, 'removed': before - len(df), 'new_count': len(df)}


def cap_outliers(df, column):
    """Cap outliers to IQR bounds (Winsorization)."""
    df = df.copy()
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    capped = ((df[column] < lower) | (df[column] > upper)).sum()
    df[column] = df[column].clip(lower=lower, upper=upper)
    return df, {'column': column, 'capped_count': int(capped), 'lower_bound': lower, 'upper_bound': upper}

## SECTION 12: Outlier Visualizations

In [12]:
def plot_boxplots(df):
    """Generate Plotly boxplots for all numerical columns."""
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    plots = {}
    for col in num_cols:
        fig = go.Figure()
        fig.add_trace(go.Box(
            y=df[col].dropna(),
            name=col,
            boxmean=True,
            marker_color='#6366f1',
            line_color='#4f46e5'
        ))
        fig.update_layout(
            title=f'Boxplot: {col}',
            yaxis_title=col,
            template='plotly_dark',
            height=350
        )
        plots[col] = fig
    return plots

## SECTION 13: Skewness Analysis

In [13]:
def calculate_skewness(df):
    """Calculate skewness for all numerical columns."""
    result = []
    for col in df.select_dtypes(include=[np.number]).columns:
        skew = df[col].skew()
        if abs(skew) < 0.5:
            classification = 'Normal'
        elif abs(skew) < 1.0:
            classification = 'Moderately Skewed'
        else:
            classification = 'Highly Skewed'
        result.append({'column': col, 'skewness': round(skew, 4), 'classification': classification})
    return pd.DataFrame(result)


def apply_log_transform(df, column):
    """Apply log transformation (shift to handle zeros/negatives)."""
    df = df.copy()
    shift = abs(df[column].min()) + 1 if df[column].min() <= 0 else 0
    df[column + '_log'] = np.log(df[column] + shift)
    return df


def apply_sqrt_transform(df, column):
    """Apply square root transformation."""
    df = df.copy()
    shift = abs(df[column].min()) if df[column].min() < 0 else 0
    df[column + '_sqrt'] = np.sqrt(df[column] + shift)
    return df


def apply_boxcox_transform(df, column):
    """Apply Box-Cox transformation."""
    df = df.copy()
    series = df[column].dropna()
    shift = abs(series.min()) + 1 if series.min() <= 0 else 0
    try:
        transformed, _ = boxcox(series + shift)
        df[column + '_boxcox'] = np.nan
        df.loc[series.index, column + '_boxcox'] = transformed
    except Exception as e:
        print(f'BoxCox failed for {column}: {e}')
    return df


def plot_skewness_charts(df):
    """Generate skewness bar chart."""
    skew_df = calculate_skewness(df)
    if skew_df.empty:
        return None
    color_map = {'Normal': '#22c55e', 'Moderately Skewed': '#f59e0b', 'Highly Skewed': '#ef4444'}
    skew_df['color'] = skew_df['classification'].map(color_map)
    fig = go.Figure(go.Bar(
        x=skew_df['column'], y=skew_df['skewness'],
        marker_color=skew_df['color'],
        text=skew_df['classification'], textposition='outside'
    ))
    fig.update_layout(title='Skewness by Column', template='plotly_dark', height=400,
                      yaxis_title='Skewness Value', xaxis_title='Column')
    return fig

## SECTION 14: Distribution Analysis

In [14]:
def plot_distributions(df):
    """Generate histogram + KDE plots for all numerical columns."""
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    plots = {}
    for col in num_cols:
        try:
            data = df[col].dropna()
            fig = go.Figure()
            fig.add_trace(go.Histogram(
                x=data, name='Histogram', nbinsx=30,
                marker_color='rgba(99,102,241,0.6)', opacity=0.75
            ))
            # KDE
            kde = stats.gaussian_kde(data)
            x_range = np.linspace(data.min(), data.max(), 200)
            kde_y = kde(x_range) * len(data) * (data.max() - data.min()) / 30
            fig.add_trace(go.Scatter(
                x=x_range, y=kde_y, mode='lines', name='KDE',
                line=dict(color='#f59e0b', width=2)
            ))
            fig.update_layout(title=f'Distribution: {col}', template='plotly_dark', height=350)
            plots[col] = fig
        except Exception as e:
            print(f'Could not plot {col}: {e}')
    return plots

## SECTION 15: Statistical Summary

In [15]:
def descriptive_statistics(df):
    """Return comprehensive descriptive statistics for all numerical columns."""
    num_df = df.select_dtypes(include=[np.number])
    if num_df.empty:
        return pd.DataFrame()
    result = []
    for col in num_df.columns:
        series = num_df[col].dropna()
        mode_val = series.mode().iloc[0] if not series.mode().empty else np.nan
        result.append({
            'Column': col,
            'Count': len(series),
            'Mean': round(series.mean(), 4),
            'Median': round(series.median(), 4),
            'Mode': round(mode_val, 4),
            'Std': round(series.std(), 4),
            'Variance': round(series.var(), 4),
            'Min': round(series.min(), 4),
            'Max': round(series.max(), 4),
            'Q1': round(series.quantile(0.25), 4),
            'Q3': round(series.quantile(0.75), 4),
            'IQR': round(series.quantile(0.75) - series.quantile(0.25), 4),
            'Skewness': round(series.skew(), 4),
            'Kurtosis': round(series.kurtosis(), 4)
        })
    return pd.DataFrame(result)

## SECTION 16: Correlation Analysis (No Heatmap)

In [16]:
def correlation_bar_chart(df, method='pearson', top_n=10):
    """Generate correlation bar charts (one per numerical column). NO heatmap."""
    num_df = df.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return {}
    corr_matrix = num_df.corr(method=method)
    plots = {}
    for col in corr_matrix.columns:
        corr_vals = corr_matrix[col].drop(col).dropna().sort_values(key=abs, ascending=False).head(top_n)
        if corr_vals.empty:
            continue
        colors = ['#22c55e' if v > 0 else '#ef4444' for v in corr_vals.values]
        fig = go.Figure(go.Bar(
            x=corr_vals.values,
            y=corr_vals.index,
            orientation='h',
            marker_color=colors,
            text=[f'{v:.3f}' for v in corr_vals.values],
            textposition='outside'
        ))
        fig.update_layout(
            title=f'{method.capitalize()} Correlation with "{col}"',
            xaxis_title='Correlation Coefficient',
            yaxis_title='Feature',
            template='plotly_dark',
            height=max(300, len(corr_vals) * 40 + 100),
            xaxis=dict(range=[-1, 1])
        )
        plots[col] = fig
    return plots

## SECTION 17: Data Quality Score

In [17]:
def calculate_data_quality_score(df):
    """Calculate 0-100 data quality score."""
    scores = {}

    # Missing values (30 pts)
    missing_pct = df.isnull().sum().sum() / df.size * 100
    scores['missing'] = max(0, 30 - missing_pct * 0.6)

    # Duplicates (20 pts)
    dup_pct = df.duplicated().sum() / len(df) * 100
    scores['duplicates'] = max(0, 20 - dup_pct * 0.4)

    # Outliers (20 pts)
    outlier_info = detect_outliers_iqr(df)
    total_outlier_pct = np.mean([v['pct'] for v in outlier_info.values()]) if outlier_info else 0
    scores['outliers'] = max(0, 20 - total_outlier_pct * 0.4)

    # Invalid values (15 pts)
    invalid = detect_invalid_values(df)
    invalid_pct = sum(v['count'] for v in invalid.values()) / len(df) * 100 if invalid else 0
    scores['invalid'] = max(0, 15 - invalid_pct * 0.3)

    # Consistency: column type uniformity (15 pts)
    scores['consistency'] = 15  # baseline; lower if mixed types detected

    total = sum(scores.values())
    return round(min(100, total), 1), scores


def plot_quality_gauge(score):
    """Generate gauge chart for data quality score."""
    if score >= 80:
        color = '#22c55e'
    elif score >= 60:
        color = '#f59e0b'
    else:
        color = '#ef4444'

    fig = go.Figure(go.Indicator(
        mode='gauge+number+delta',
        value=score,
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': 'Data Quality Score', 'font': {'size': 20}},
        gauge={
            'axis': {'range': [0, 100], 'tickwidth': 1},
            'bar': {'color': color},
            'steps': [
                {'range': [0, 40], 'color': '#3f1414'},
                {'range': [40, 70], 'color': '#3f2a00'},
                {'range': [70, 100], 'color': '#0d2e0d'}
            ],
            'threshold': {'line': {'color': 'white', 'width': 4}, 'thickness': 0.75, 'value': score}
        }
    ))
    fig.update_layout(template='plotly_dark', height=300)
    return fig

## SECTION 18: Export Functions

In [18]:
def export_csv(df):
    """Export DataFrame as CSV bytes."""
    return df.to_csv(index=False).encode('utf-8')


def export_excel(df):
    """Export DataFrame as Excel bytes."""
    output = io.BytesIO()
    with pd.ExcelWriter(output, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name='Processed_Data')
    return output.getvalue()

## SECTION 19: Save Processing History

In [19]:
def save_operation(file_name, operation, details):
    """Save a preprocessing operation to the SQLite processing history table."""
    try:
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO processing_history (file_name, operation, details, timestamp)
            VALUES (?, ?, ?, ?)
        ''', (file_name, operation, str(details), datetime.now().isoformat()))
        conn.commit()
        conn.close()
    except Exception as e:
        print(f'Error saving operation: {e}')


def get_processing_history(file_name=None):
    """Retrieve processing history from database."""
    try:
        conn = sqlite3.connect(DB_NAME)
        if file_name:
            df = pd.read_sql('SELECT * FROM processing_history WHERE file_name=? ORDER BY timestamp DESC', conn, params=(file_name,))
        else:
            df = pd.read_sql('SELECT * FROM processing_history ORDER BY timestamp DESC', conn)
        conn.close()
        return df
    except Exception as e:
        print(f'Error fetching history: {e}')
        return pd.DataFrame()


print('All functions loaded successfully.')

All functions loaded successfully.
